# Manhattan data for the Day 4 experiment

Run one cell at a time, top to bottom. Each one prints what it found and saves to `data/processed`.

## Step 1: setup (takes a few seconds)

In [ ]:
import json
import geopandas as gpd, pandas as pd, requests, shapely
from shapely.geometry import Polygon
from mapkit.paths import PROCESSED

def nyc_open_data(dataset, where=None):
    """Download one NYC Open Data table."""
    params = {"$limit": 300000}
    if where:
        params["$where"] = where
    r = requests.get(f"https://data.cityofnewyork.us/resource/{dataset}.geojson", params=params, timeout=600)
    r.raise_for_status()
    features = [x for x in r.json()["features"] if x.get("geometry")]
    return gpd.GeoDataFrame.from_features(features, crs=4326)

print("Ready. Saving to", PROCESSED)

## Step 2: Manhattan's outline

In [ ]:
f = PROCESSED / "nyc_boroughs.geojson"
if not f.exists():
    official = "https://services5.arcgis.com/GfwWNkhOj9bNBqoJ/arcgis/rest/services/NYC_Borough_Boundary/FeatureServer/0/query"
    backup = "https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/new-york-city-boroughs.geojson"
    try:
        r = requests.get(official, params={"where": "1=1", "outFields": "*", "outSR": 4326, "f": "geojson"}, timeout=120)
        assert len(r.json()["features"]) == 5
        source = "official"
    except Exception:
        print("City Planning download failed, using the backup copy")
        r = requests.get(backup, timeout=120)
        source = "backup"
    f.write_bytes(r.content)
    (PROCESSED / "nyc_boroughs_source.txt").write_text(source)

boroughs = gpd.read_file(f)
if not boroughs.geometry.is_valid.all():
    # The city's file packs every island into one shape as a list of rings. Rebuild real land, with lakes as holes.
    def fix_rings(geom):
        rings = sorted((Polygon(r).buffer(0) for r in shapely.get_rings(geom)), key=lambda p: -p.area)
        out = rings[0]
        for r in rings[1:]:
            out = out.symmetric_difference(r)
        return out
    boroughs["geometry"] = [fix_rings(x) for x in boroughs.geometry]
    boroughs.to_file(f, driver="GeoJSON")
label = next(c for c in boroughs.columns if boroughs[c].astype(str).str.contains("Manhattan").any())
island = boroughs[boroughs[label].astype(str).str.contains("Manhattan")].union_all()
print("Borough source:", (PROCESSED / "nyc_boroughs_source.txt").read_text())
print(f"Manhattan: {gpd.GeoSeries([island], crs=4326).to_crs(32618).area.iloc[0] / 1e6:.1f} km2 (should be about 59)")
gpd.GeoSeries([island], crs=4326).plot(figsize=(3, 6));

## Step 3: parks (about a minute)

In [ ]:
parks = nyc_open_data("enfh-gkve")
parks = parks[parks.intersects(island) & parks.geom_type.isin(["Polygon", "MultiPolygon"])]
print(len(parks), "parks. Columns:", list(parks.columns))
assert len(parks) >= 50, "Too few parks: stop here and send Claude a screenshot"
keep = [c for c in ("signname", "acres", "typecategory", "geometry") if c in parks.columns]
parks[keep].to_file(PROCESSED / "manhattan_parks.geojson", driver="GeoJSON")
print("Largest:", parks.sort_values("acres", key=lambda s: pd.to_numeric(s, errors="coerce"), ascending=False)["signname"].head(5).tolist() if {"signname", "acres"} <= set(parks.columns) else "(names not found)")
parks.plot(color="green", figsize=(3, 6));

## Step 4: streets (a few minutes, this is the slow one)

In [ ]:
try:
    streets = nyc_open_data("inkn-q76z", where="boroughcode='1'")
except Exception as e:
    print(f"Manhattan-only request failed ({type(e).__name__}), downloading all five boroughs instead")
    streets = nyc_open_data("inkn-q76z")
streets = streets[streets.intersects(island) & streets.geom_type.isin(["LineString", "MultiLineString"])].copy()
print(len(streets), "street segments. Columns:", list(streets.columns))
assert len(streets) >= 1000, "Too few streets: stop here and send Claude a screenshot"
name = next((c for c in ("full_street_name", "full_stree", "stname_label", "stname_lab", "street_name", "street_nam") if c in streets.columns), None)
streets["street"] = streets[name] if name else ""
keep = [c for c in ("street", "rw_type", "geometry") if c in streets.columns]
streets[keep].to_file(PROCESSED / "manhattan_streets.geojson", driver="GeoJSON")
print("Avenues found:", int(streets.street.fillna("").str.contains("AVENUE|BROADWAY", case=False).sum()))
streets.plot(color="black", linewidth=0.2, figsize=(3, 6));

## Step 5: note the sources, so the map credits them correctly

In [ ]:
report_file = PROCESSED / "fetch_report.json"
report = json.loads(report_file.read_text()) if report_file.exists() else {}
report["day4_manhattan"] = {"status": "ok", "parks_source": "NYC Parks", "streets_source": "NYC Street Centerline", "parks": len(parks), "streets": len(streets)}
report_file.write_text(json.dumps(report, indent=2))
print("All done. Now run day04b_clusters_manhattan.")